# Parallel Trends Analysis
## Section 232 Tariff Impact Study

This notebook tests the **parallel trends assumption** required for valid difference-in-differences estimation.

**Key Question:** Did treated and control groups exhibit parallel trends in the pre-treatment period?

### Approach:
1. Visual inspection of pre-treatment trends
2. Formal statistical test (treated × time interaction)
3. Pre-treatment R² and goodness-of-fit
4. Placebo tests on pre-period

In [ ]:
import sys
sys.path.append('../models')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from datetime import datetime

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

%matplotlib inline
%load_ext autoreload
%autoreload 2

## 1. Load Data

In [ ]:
# Load cleaned panel data
data_path = '../data/clean/steel_panel.parquet'
df = pd.read_parquet(data_path)

# Display basic info
print(f"Observations: {len(df):,}")
print(f"Countries: {df['country'].nunique()}")
print(f"Products: {df['product'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

df.head()

## 2. Define Treatment Groups

In [ ]:
# Treatment date: Section 232 implementation
TREATMENT_DATE = pd.to_datetime('2018-03-23')

# Define treated countries (subject to 25% tariff)
# Exempt: Canada, Mexico (initially)
EXEMPT_COUNTRIES = ['Canada', 'Mexico']
treated_countries = [c for c in df['country'].unique() if c not in EXEMPT_COUNTRIES]

# Create treatment indicators
df['treated'] = df['country'].isin(treated_countries).astype(int)
df['post'] = (df['date'] >= TREATMENT_DATE).astype(int)

print(f"Treated countries: {len(treated_countries)}")
print(f"Control countries: {len(EXEMPT_COUNTRIES)}")

# Treatment group summary
df.groupby('treated')['country'].nunique()

## 3. Visual Inspection: Pre-Treatment Trends

Plot average outcomes for treated vs. control groups over time

In [ ]:
# Aggregate to monthly level by treatment status
trends = df.groupby(['date', 'treated']).agg({
    'import_value': 'sum',
    'import_quantity': 'sum'
}).reset_index()

# Create log values
trends['log_value'] = np.log(trends['import_value'] + 1)
trends['log_quantity'] = np.log(trends['import_quantity'] + 1)

# Separate treated and control
treated_trends = trends[trends['treated'] == 1]
control_trends = trends[trends['treated'] == 0]

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Panel A: Import Value
axes[0].plot(treated_trends['date'], treated_trends['log_value'], 
             'o-', label='Treated', linewidth=2, markersize=3)
axes[0].plot(control_trends['date'], control_trends['log_value'], 
             's-', label='Control', linewidth=2, markersize=3, alpha=0.7)
axes[0].axvline(TREATMENT_DATE, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Tariff')
axes[0].set_ylabel('Log Import Value', fontsize=12)
axes[0].set_title('Parallel Trends Test: Import Value', fontsize=14, fontweight='bold')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# Panel B: Import Quantity
axes[1].plot(treated_trends['date'], treated_trends['log_quantity'], 
             'o-', label='Treated', linewidth=2, markersize=3)
axes[1].plot(control_trends['date'], control_trends['log_quantity'], 
             's-', label='Control', linewidth=2, markersize=3, alpha=0.7)
axes[1].axvline(TREATMENT_DATE, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Tariff')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Log Import Quantity', fontsize=12)
axes[1].set_title('Parallel Trends Test: Import Quantity', fontsize=14, fontweight='bold')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../viz/parallel_trends_visual.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Formal Statistical Test

Test whether treated and control groups had parallel trends in pre-period.

**Specification:**
```
Y_it = α + β₁·Treated_i + β₂·Time_t + β₃·(Treated_i × Time_t) + ε_it
```

**Null hypothesis:** β₃ = 0 (parallel trends)

If we reject (p < 0.05), trends were NOT parallel.

In [ ]:
# Filter to pre-treatment period
pre_data = df[df['post'] == 0].copy()

# Create time trend (months from start)
pre_data['time_trend'] = (pre_data['date'] - pre_data['date'].min()).dt.days / 30

# Create log outcome
pre_data['log_value'] = np.log(pre_data['import_value'] + 1)

# Interaction term
pre_data['treated_x_trend'] = pre_data['treated'] * pre_data['time_trend']

# Run regression
formula = 'log_value ~ treated + time_trend + treated_x_trend + C(product) + C(country)'
model = smf.ols(formula, data=pre_data)
results = model.fit(cov_type='cluster', cov_kwds={'groups': pre_data['country']})

print("\n" + "="*60)
print("PARALLEL TRENDS TEST (Pre-Treatment Period)")
print("="*60)
print(f"\nCoefficient on Treated × Time: {results.params['treated_x_trend']:.6f}")
print(f"Standard Error (clustered):     {results.bse['treated_x_trend']:.6f}")
print(f"T-statistic:                    {results.tvalues['treated_x_trend']:.3f}")
print(f"P-value:                        {results.pvalues['treated_x_trend']:.4f}")
print(f"\nR-squared:                      {results.rsquared:.4f}")
print(f"N (pre-treatment obs):          {results.nobs:,.0f}")

if results.pvalues['treated_x_trend'] < 0.05:
    print("\n⚠️  WARNING: Parallel trends assumption is VIOLATED (p < 0.05)")
    print("   Consider alternative identification strategies or specifications.")
else:
    print("\n✓  Parallel trends assumption is SATISFIED (p ≥ 0.05)")
    print("   Proceed with DID estimation.")

print("="*60)

## 5. Pre-Treatment Fit Quality

Assess how well the pre-treatment model fits the data.

In [ ]:
# Predict in-sample
pre_data['predicted'] = results.predict()
pre_data['residual'] = pre_data['log_value'] - pre_data['predicted']

# Calculate RMSE by group
rmse_treated = np.sqrt(np.mean(pre_data[pre_data['treated']==1]['residual']**2))
rmse_control = np.sqrt(np.mean(pre_data[pre_data['treated']==0]['residual']**2))

print("Pre-Treatment Fit Quality:")
print(f"  RMSE (treated): {rmse_treated:.4f}")
print(f"  RMSE (control): {rmse_control:.4f}")
print(f"  Overall R²:     {results.rsquared:.4f}")

# Plot residuals
fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(pre_data['predicted'], pre_data['residual'], alpha=0.3, s=10)
ax.axhline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Fitted Values', fontsize=12)
ax.set_ylabel('Residuals', fontsize=12)
ax.set_title('Residual Plot: Pre-Treatment Period', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Placebo Test: Fake Treatment Dates

Test whether we detect "effects" at fake treatment dates in the pre-period.
We should NOT find effects if parallel trends hold.

In [ ]:
# Define fake treatment dates (in pre-period)
fake_dates = pd.date_range(start='2014-01-01', end='2017-12-01', freq='6MS')

placebo_results = []

for fake_date in fake_dates:
    # Create fake post indicator
    pre_data['fake_post'] = (pre_data['date'] >= fake_date).astype(int)
    pre_data['fake_did'] = pre_data['treated'] * pre_data['fake_post']
    
    # Run DID regression
    formula = 'log_value ~ fake_did + C(product) + C(country) + C(date)'
    model = smf.ols(formula, data=pre_data)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': pre_data['country']})
    
    placebo_results.append({
        'fake_date': fake_date,
        'coefficient': res.params['fake_did'],
        'std_error': res.bse['fake_did'],
        'p_value': res.pvalues['fake_did']
    })

placebo_df = pd.DataFrame(placebo_results)
placebo_df['ci_lower'] = placebo_df['coefficient'] - 1.96 * placebo_df['std_error']
placebo_df['ci_upper'] = placebo_df['coefficient'] + 1.96 * placebo_df['std_error']

# Plot placebo coefficients
fig, ax = plt.subplots(figsize=(12, 6))
ax.errorbar(placebo_df['fake_date'], placebo_df['coefficient'], 
            yerr=1.96*placebo_df['std_error'], fmt='o', capsize=5, capthick=2, linewidth=2)
ax.axhline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Fake Treatment Date', fontsize=12)
ax.set_ylabel('Placebo DID Coefficient', fontsize=12)
ax.set_title('Placebo Test: Fake Treatment Dates (Should be ~0)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../viz/placebo_test.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPlacebo Test Results:")
print(placebo_df.to_string(index=False))
print(f"\nSignificant placebos (p < 0.05): {(placebo_df['p_value'] < 0.05).sum()} / {len(placebo_df)}")

## 7. Summary and Recommendations

### Findings:
1. **Visual inspection:** [Describe what you observe]
2. **Statistical test:** [Report p-value and conclusion]
3. **Placebo tests:** [Number of significant placebos]

### Recommendation:
- [ ] Parallel trends assumption appears satisfied → Proceed with DID
- [ ] Concerns about parallel trends → Consider alternative specs or synthetic control
- [ ] Strong violations → DID may not be appropriate

In [ ]:
# Save summary report
summary_report = {
    'parallel_trends_test': {
        'coefficient': float(results.params['treated_x_trend']),
        'p_value': float(results.pvalues['treated_x_trend']),
        'satisfied': bool(results.pvalues['treated_x_trend'] >= 0.05)
    },
    'fit_quality': {
        'r_squared': float(results.rsquared),
        'rmse_treated': float(rmse_treated),
        'rmse_control': float(rmse_control)
    },
    'placebo_tests': {
        'n_tests': len(placebo_df),
        'n_significant': int((placebo_df['p_value'] < 0.05).sum()),
        'max_coefficient': float(placebo_df['coefficient'].abs().max())
    }
}

import json
with open('../reports/parallel_trends_summary.json', 'w') as f:
    json.dump(summary_report, f, indent=2)

print("\n✓ Summary report saved to ../reports/parallel_trends_summary.json")